# Week 4 · Day 4 — Cortex: run an LLM **inside** Snowflake

*You've stored, loaded, and analyzed matters. The finale: summarize and classify them without ever leaving SQL.*

**By the end you'll have shipped:** a **`matter_intelligence`** table where every matter carries an **LLM-written summary** and a **classification** — produced by Snowflake **Cortex** functions called right in a `SELECT`. This is the capstone slice: *store → summarize → classify, in the warehouse.*

### 📋 Lesson card

| | |
|---|---|
| **Module** | M4 · Data & Snowflake → M2 · Building with Claude (Week 4) |
| **Prerequisites** | W4D1–D3 (tables, loading, analytical SQL) |
| **Est. time** | ~30 min |
| **Capstone slice** | **The core of *Matter Intelligence*** — summarize + classify matters in place |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — Cortex is **mocked** locally (just like the Claude lessons mock the API) |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain what **Snowflake Cortex** is — LLM functions you call in SQL.
- Use **`SUMMARIZE`** and **`CLASSIFY_TEXT`** on a text column, per row.
- **Enrich a whole table** with LLM output in one query and persist it (CTAS).
- Compare **Cortex (in-warehouse)** vs. the **Claude API (in your code)** — a buy/where-to-run decision.
- Apply **responsible-AI** habits: verify output, respect privilege/PII, keep a human in the loop.

### ⚖️ Why it matters

*Matter Intelligence* needs to read each matter and produce a **plain-English summary** and a **category**. You could pull every row into Python and call the Claude API — or you could do it **where the data already lives**. **Cortex** runs Claude-class models *inside Snowflake*, callable from SQL: no data leaves the warehouse, and one `SELECT` enriches a whole table. That's a genuinely different architecture, and today you build it.

### ⚙️ Setup

Loads the `matters` table, gives each matter a short **`notes`** text to work on, and registers two functions — **`ai_summarize`** and **`ai_classify`** — that stand in for Cortex offline.

> On a real Snowflake account these are `SNOWFLAKE.CORTEX.SUMMARIZE(...)` and `SNOWFLAKE.CORTEX.CLASSIFY_TEXT(...)`, running server-side. Offline we register **mock** versions so the exact same SQL runs with no account and no key — the same mock-first pattern as the Claude lessons.

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'matters': [{'matter_id': 'M-1001', 'client': 'Acme Corp', 'practice_area': 'Contracts', 'status': 'Active', 'open_date': '2026-01-12', 'amount_billed': 18500.0, 'is_privileged': True, 'lead_attorney': 'R. Nguyen'}, {'matter_id': 'M-1002', 'client': 'Brightline LLC', 'practice_area': 'Litigation', 'status': 'Active', 'open_date': '2025-11-03', 'amount_billed': 42750.5, 'is_privileged': True, 'lead_attorney': 'S. Patel'}, {'matter_id': 'M-1003', 'client': 'Cedar Holdings', 'practice_area': 'M&A', 'status': 'Closed', 'open_date': '2025-06-21', 'amount_billed': 131200.0, 'is_privileged': True, 'lead_attorney': 'R. Nguyen'}, {'matter_id': 'M-1004', 'client': 'Dovetail Inc', 'practice_area': 'Employment', 'status': 'Active', 'open_date': '2026-02-15', 'amount_billed': 9800.0, 'is_privileged': False, 'lead_attorney': 'T. Alvarez'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

In [ ]:
# 1) Give every matter a short notes text to summarize/classify (built in SQL from its own fields).
try:
    run_sql("ALTER TABLE matters ADD COLUMN notes STRING")
except Exception:
    pass  # column already exists on a re-run
run_sql("""
    UPDATE matters SET notes =
        'This ' || LOWER(practice_area) || ' matter for ' || client ||
        ' (' || matter_id || ') is currently ' || LOWER(status) ||
        '. Amount billed to date: $' || CAST(amount_billed AS STRING) || '.'
""")

# 2) Register Cortex stand-ins: real on Snowflake, mocked on DuckDB.
if BACKEND == "duckdb":
    def _ai_summarize(t):
        if t is None:
            return None
        return str(t).split(".")[0].strip() + "."          # a plausible one-line summary
    def _ai_classify(t):
        s = str(t or "").lower()
        if any(w in s for w in ("litigation", "dispute", "employment", "claim")):
            return "contentious"
        if any(w in s for w in ("contract", "m&a", "acquisition", "merger", "transaction")):
            return "transactional"
        return "advisory"
    for name, fn in (("ai_summarize", _ai_summarize), ("ai_classify", _ai_classify)):
        try:
            _con.create_function(name, fn, ["VARCHAR"], "VARCHAR")
        except Exception:
            pass  # already registered
else:
    # Real Cortex, wrapped so the teaching SQL below is identical on both backends.
    _con.cursor().execute("CREATE OR REPLACE FUNCTION ai_summarize(t STRING) RETURNS STRING AS $$ SNOWFLAKE.CORTEX.SUMMARIZE(t) $$")
    _con.cursor().execute("CREATE OR REPLACE FUNCTION ai_classify(t STRING) RETURNS STRING AS $$ SNOWFLAKE.CORTEX.CLASSIFY_TEXT(t, ['contentious','transactional','advisory']):label::string $$")

print(f"✅ Cortex stand-ins ready on {BACKEND.upper()}")
run_sql("SELECT matter_id, notes FROM matters LIMIT 2")

### 1 · What is Cortex?

**Snowflake Cortex** is a set of built-in **LLM functions** you call from SQL — no API keys in your code, no data leaving the warehouse. The headline ones:

| Cortex function | What it does |
|---|---|
| **`SUMMARIZE(text)`** | a plain-English summary |
| **`CLASSIFY_TEXT(text, [labels])`** | pick the best-fitting label |
| **`COMPLETE(model, prompt)`** | free-form generation (choose the model) |
| **`SENTIMENT(text)`** | a −1…+1 sentiment score |
| **`EXTRACT_ANSWER(text, question)`** | pull an answer from a passage |

The mental model: **the same thing the Claude API does, but invoked in a `SELECT` and run next to your data.** Today we use summarize + classify.

### 2 · `SUMMARIZE` — one summary per matter

Call it like any function, on the `notes` column. Each row gets its own summary.

*On Snowflake you'd write `SNOWFLAKE.CORTEX.SUMMARIZE(notes)`; here `ai_summarize(notes)` is the local stand-in.*

In [ ]:
run_sql("""
    SELECT matter_id, client,
           ai_summarize(notes) AS summary        -- Snowflake: SNOWFLAKE.CORTEX.SUMMARIZE(notes)
    FROM matters
    ORDER BY matter_id
    LIMIT 5
""")

**What just happened:** the model ran **once per row**, right in the query — no Python loop, no data export. On Snowflake this is a real LLM summarizing each matter; offline it's our deterministic stand-in so the notebook runs anywhere.

### 3 · `CLASSIFY_TEXT` — bucket each matter

Classification maps each matter's text to one of a small set of labels you provide — here, coarse buckets (`contentious` / `transactional` / `advisory`) above the fine-grained `practice_area`.

*On Snowflake: `SNOWFLAKE.CORTEX.CLASSIFY_TEXT(notes, ['contentious','transactional','advisory'])`.*

In [ ]:
run_sql("""
    SELECT matter_id, practice_area,
           ai_classify(notes) AS matter_type    -- Snowflake: CLASSIFY_TEXT(notes, [...labels...])
    FROM matters
    ORDER BY matter_type, matter_id
""")

### 4 · Enrich the whole table, then persist it (CTAS)

Because Cortex is *just SQL*, you enrich an entire table and **save the result** in one statement (`CREATE TABLE AS SELECT`, from W3D4/W4D1). This materialized table is what an API or dashboard would read.

In [ ]:
run_sql("""
    CREATE OR REPLACE TABLE matter_intelligence AS
        SELECT matter_id,
               client,
               practice_area,
               status,
               amount_billed,
               ai_classify(notes)  AS matter_type,
               ai_summarize(notes) AS summary
        FROM matters
""")
run_sql("SELECT matter_id, client, matter_type, summary FROM matter_intelligence ORDER BY amount_billed DESC LIMIT 6")

**What just happened:** you shipped the capstone's heart — a `matter_intelligence` table where every matter carries an LLM **summary** and **type**, produced in-warehouse. Now analytics compose on top:

In [ ]:
run_sql("""
    SELECT matter_type,
           COUNT(*)                     AS matters,
           ROUND(SUM(amount_billed), 2) AS total_billed
    FROM matter_intelligence
    GROUP BY matter_type
    ORDER BY total_billed DESC
""")

> **`Go Deeper 🔧` — `COMPLETE` for anything, and picking a model.** `SNOWFLAKE.CORTEX.COMPLETE('claude-...', prompt)` runs a free-form prompt, so you can extract structured fields, draft language, or flag risk — the same prompting skills you'll learn in the Claude module, invoked in SQL. You pass the **model name**, so you choose the speed/quality trade-off per query (fast model for bulk classification, a stronger one for nuanced drafting).

### 5 · Cortex vs. the Claude API — where should the LLM run?

Same capability, two homes — a real architecture decision:

| | **Cortex** (in Snowflake) | **Claude API** (in your code) |
|---|---|---|
| Where it runs | in the warehouse, next to the data | in your app / notebook |
| Data movement | none — data never leaves | you send text to the API |
| Called from | **SQL** (`SELECT ...`) | Python (`client.messages.create`) |
| Best for | bulk enrichment of stored tables | interactive apps, custom prompting/tools |
| Governance | inherits Snowflake's access controls | you manage keys, egress, logging |

Often you'll use **both**: Cortex to enrich tables at scale in place, the Claude API for the interactive, tool-using parts of the app. (And the "should we build any of this?" lens — Harvey — comes at the end of the course.)

> **`⚖️ Responsible AI` — non-negotiable for legal work.**
> - **Verify.** LLM summaries can be wrong or omit a key fact. A lawyer reviews anything relied upon — treat `summary`/`matter_type` as a **draft**, not truth.
> - **Privilege & PII.** Cortex keeps data in Snowflake (a governance plus), but the same access controls and privilege rules still apply — don't widen who can read a summary of a privileged matter.
> - **Keep the human in the loop.** These functions accelerate review; they don't replace judgment. Log what model produced what, so outputs are auditable.

### ✍️ Your turn

In [ ]:
# TODO 1: SELECT matter_id and ai_summarize(notes) for only the ACTIVE matters

# TODO 2: count how many matters fall into each ai_classify(notes) bucket

# TODO 3: build a table `active_intelligence` (CTAS) = active matters + summary + matter_type


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    SELECT matter_id, ai_summarize(notes) AS summary
    FROM matters
    WHERE status = 'Active'
""")

# 2
run_sql("""
    SELECT ai_classify(notes) AS matter_type, COUNT(*) AS n
    FROM matters
    GROUP BY matter_type
    ORDER BY n DESC
""")

# 3
run_sql("""
    CREATE OR REPLACE TABLE active_intelligence AS
        SELECT matter_id, client, amount_billed,
               ai_classify(notes)  AS matter_type,
               ai_summarize(notes) AS summary
        FROM matters
        WHERE status = 'Active'
""")
run_sql("SELECT * FROM active_intelligence ORDER BY amount_billed DESC")
```
</details>

### 🚀 Build the artifact — the `matter_intelligence` table

You've shipped the capstone's core: a governed, queryable table where every matter carries an **LLM summary** and a **classification**, built entirely in Snowflake SQL. Every later slice — the FastAPI endpoint, the React UI — reads from exactly this.

In [ ]:
brief = run_sql("SELECT matter_id, client, matter_type, summary FROM matter_intelligence ORDER BY matter_id")
print(f"📦 Shipped: matter_intelligence — {len(brief)} matters, each summarized + classified in-warehouse.")
brief.head(8)

> **🔗 Your world.** This is *Matter Intelligence*, the data layer done: real matters, stored and analyzed in Snowflake, enriched by an LLM **without the data ever leaving**. Swap the mock for real Cortex (add credentials to `.env`) and the identical SQL runs Claude-class models over the firm's matters. Next the course serves this table through an API and a UI.

### 🌱 A look ahead — is it time for **dbt**? (Week 5, Day 1)

Look back at Weeks 3–4: you wrote a lot of `CREATE OR REPLACE TABLE ... AS SELECT` — `active_matters`, `category_summary`, `matter_intelligence`. Each is a **transformation** turning raw tables into curated ones. Right now they live as loose SQL in notebooks: no version control, no tests, no picture of what depends on what.

**That's the exact problem [dbt](https://docs.getdbt.com/) solves** — it turns those `SELECT`s into a **versioned, tested, documented** project with automatic **lineage** (which table feeds which). We deliberately learn it **now, not earlier**: dbt is a *wrapper around SQL + a warehouse*, so it only clicks once you can comfortably write the `JOIN`s and `GROUP BY`s underneath — which you now can.

**So: not "learn dbt too" — learn dbt *next.*** Week 5 Day 1 introduces it with the `dbt-duckdb` adapter (offline, same stand-in as this week), rebuilding `matter_intelligence` as a proper dbt model with a test. *Nothing to install yet — it's the next lesson.*

### 📝 Recap — what you shipped

- **Cortex** = LLM functions (`SUMMARIZE`, `CLASSIFY_TEXT`, `COMPLETE`, ...) callable **in SQL**, run inside Snowflake.
- You summarized and classified matters **per row**, then enriched and **persisted** a whole table with CTAS.
- **Cortex vs. Claude API** is a where-to-run-the-LLM decision — often you use both.
- **Responsible AI:** verify output, respect privilege/PII, keep a human in the loop.
- **Artifact:** `matter_intelligence` — the capstone's summarized-and-classified core.

### 🧠 Check your understanding

1. What is Snowflake Cortex, and how do you call it?
2. Name one architectural advantage of Cortex over calling the Claude API from your code.
3. Why enrich a table with CTAS instead of recomputing the LLM output every query?
4. Give two responsible-AI rules that apply to an LLM-written matter summary.

<details><summary>✅ Answers</summary>

1. Built-in **LLM functions** (summarize, classify, complete, ...) you invoke **from SQL** (`SELECT SNOWFLAKE.CORTEX.SUMMARIZE(col) ...`), running inside Snowflake.
2. **No data movement** — the text never leaves the warehouse, so it inherits Snowflake's governance/access controls (and there are no API keys in your app).
3. LLM calls cost time/money; CTAS **materializes** the results once so downstream queries and dashboards read a stored table instead of re-running the model.
4. Any two: **verify** output (a lawyer reviews), respect **privilege/PII** and access controls, **keep a human in the loop**, and **log** model/version for auditability.
</details>

### ➡️ Next up — Week 5, Day 1: dbt, the transformation layer

Turn this week's loose `CREATE TABLE AS SELECT` statements into a real **dbt project** — versioned models, a data test, and automatic lineage — using the offline `dbt-duckdb` adapter. We'll rebuild `matter_intelligence` the disciplined way.

*A small `dbt-duckdb` install will kick off that lesson (auto-handled) — nothing to do now.*

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| **Cortex** | Snowflake's built-in LLM functions, called in SQL |
| **`SUMMARIZE`** | LLM summary of a text column |
| **`CLASSIFY_TEXT`** | pick the best label from a provided set |
| **`COMPLETE`** | free-form generation (you pass the model) |
| **In-warehouse LLM** | the model runs next to the data; nothing is exported |
| **CTAS** | persist enriched output as a table |
| **dbt** | tool that versions/tests/documents SQL transformations (Week 5) |

**Docs:** Snowflake Cortex LLM functions — https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions · Claude models — https://platform.claude.com/docs/en/about-claude/models/overview · dbt — https://docs.getdbt.com/

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*